# PANGAEA batch metadata extraction

Loop over all CSV files under `data/experiment/pangea`, run the full pipeline with the `croissant_pangaea_standard`, and save JSON outputs under `outputs/<experiment_run>/` (one folder per model/backend run).

Set `LLM_MODEL` (and provider) in `.env` before running. `EXPERIMENT_RUN` defaults to `pangaea_<model_slug>` from the active model.

In [1]:
import json
import logging
import os
import re
import sys
from pathlib import Path

sys.path.insert(0, '..')

from dotenv import load_dotenv

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import run_metadata_extraction
from src.standards import METADATA_STANDARDS

load_dotenv(os.path.join('..', '.env'))

BASE = os.path.abspath(os.path.join('..'))
STANDARD_NAME = 'croissant_pangaea_standard'
TOPOLOGY = 'single'

LLM_NAME = get_model_name()
LLM_SLUG = re.sub(r'[^\w.\-]+', '_', LLM_NAME.replace('/', '_'))

# Folder name for this experiment run.
EXPERIMENT_RUN = 'pangaea_Qwen3.5-122B'

PANGEA_DIR = os.path.join(BASE, 'data/experiment/pangea')
if not os.path.isdir(PANGEA_DIR):
    PANGEA_DIR = os.path.join(BASE, 'data/experiment/pangaea_datasets')

OUTPUT_DIR = Path(BASE) / 'outputs' / EXPERIMENT_RUN
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

metadata_standard = METADATA_STANDARDS[STANDARD_NAME]


def output_path_for(csv_path: Path) -> Path:
    """Output file for one dataset; skip/resume is scoped to OUTPUT_DIR only."""
    return OUTPUT_DIR / f'{csv_path.stem}_pangaea.json'


def collect_pangea_csv_files(data_dir: str) -> list[Path]:
    root = Path(data_dir)
    if not root.is_dir():
        raise FileNotFoundError(f'PANGAEA data directory not found: {data_dir}')
    return sorted(root.glob('*.csv'))


all_csv_files = collect_pangea_csv_files(PANGEA_DIR)
csv_files = [p for p in all_csv_files if not output_path_for(p).exists()]
skipped_count = len(all_csv_files) - len(csv_files)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'LLM model: {LLM_NAME}')
print(f'Experiment run: {EXPERIMENT_RUN}')
print(f'Data directory: {PANGEA_DIR}')
print(f'Found {len(all_csv_files)} CSV file(s)')
print(f'Skipping {skipped_count} already completed in {OUTPUT_DIR}')
print(f'Remaining to run: {len(csv_files)}')
print(f'Output directory: {OUTPUT_DIR}')

LLM provider: surf
LLM model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Experiment run: pangaea_Qwen3.5-122B
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/pangaea_datasets
Found 75 CSV file(s)
Skipping 0 already completed in /home/com3dian/Github/metadata_agent/outputs/pangaea_Qwen3.5-122B
Remaining to run: 75
Output directory: /home/com3dian/Github/metadata_agent/outputs/pangaea_Qwen3.5-122B


In [2]:
def extract_metadata(metadata_result):
    if metadata_result is None:
        return None

    metadata = metadata_result.final_metadata
    if not metadata:
        workspace = metadata_result.final_workspace or {}
        metadata = workspace.get('metadata_output')

    # Guard against fallback/context objects; keep only schema-shaped metadata.
    if isinstance(metadata, dict) and {'name', 'description', 'keywords'}.issubset(metadata.keys()):
        return {
            'name': metadata.get('name'),
            'description': metadata.get('description'),
            'keywords': metadata.get('keywords') or [],
        }

    return None


def is_empty_metadata(metadata):
    if metadata is None:
        return True
    if isinstance(metadata, dict):
        if {'name', 'description', 'keywords'}.issubset(metadata.keys()):
            return False
        return len(metadata) == 0
    if isinstance(metadata, str):
        return not metadata.strip()
    return not metadata


def run_pipeline(csv_path: Path, attempt: int = 1):
    context_name = f'pangea_{csv_path.stem}'
    print(f'\n{"=" * 60}')
    print(f'[{attempt}] Processing {csv_path.name} (context: {context_name})')

    return run_metadata_extraction(
        source=str(csv_path.resolve()),
        metadata_standard=metadata_standard,
        metadata_standard_name=STANDARD_NAME,
        name=context_name,
        topology_name=TOPOLOGY,
    )


results_summary = []
success_count = 0

for csv_path in csv_files:
    entry = {
        'csv': csv_path.name,
        'success': False,
        'output_file': None,
        'attempts': 0,
        'error': None,
    }

    metadata = None
    for attempt in (1, 2):
        entry['attempts'] = attempt
        try:
            result = run_pipeline(csv_path, attempt=attempt)
            metadata = extract_metadata(result)
        except Exception as exc:
            entry['error'] = str(exc)
            print(f'  Attempt {attempt} failed: {exc}')
            continue

        if not is_empty_metadata(metadata):
            break

        print(f'  Attempt {attempt} returned empty metadata' + (' — retrying' if attempt == 1 else ''))

    if is_empty_metadata(metadata):
        print(f'  FAILED: no metadata for {csv_path.name}')
        results_summary.append(entry)
        continue

    output_file = output_path_for(csv_path)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    entry['success'] = True
    entry['output_file'] = str(output_file)
    success_count += 1
    print(f'  Saved: {output_file}')
    results_summary.append(entry)

print(f'\n{"=" * 60}')
print(f'Batch complete: {success_count}/{len(csv_files)} successful this run')
print(f'Previously completed (skipped): {skipped_count}')
print(f'Total with outputs: {skipped_count + success_count}/{len(all_csv_files)}')
for item in results_summary:
    status = 'OK' if item['success'] else 'FAIL'
    detail = item['output_file'] or item.get('error') or 'empty metadata'
    print(f'  [{status}] {item["csv"]} ({item["attempts"]} attempt(s)) -> {detail}')


[1] Processing 897882.csv (context: pangea_897882)


/home/com3dian/Github/metadata_agent/notebooks/../src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-02 12:09:17,130 - INFO - root - PlanExecutor initialized with topology: single
2026-07-02 12:09:17,131 - INFO - root -   Players per step: 1
2026-07-02 12:09:17,131 - INFO - root -   Debate rounds: 0
2026-07-02 12:09:17,131 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-02 12:09:17,132 - INFO - root - Orchestrator initialized with topology: single
2026-07-02 12:09:17,132 - INFO - root - ============================================================
2026-07-02 12:09:17,132 - INFO - root - STARTING ORCHESTRATION
2026-07-02 12:09:17,132 - INFO - root - Context: pangea_897882
2026-07-02 12:09:17,133 - INFO - root - Type: single_csv
2026-07-02 12:09:17,133 - INFO - root - Resources: ['897882']
2026-07-02 12:09:17,133 - INFO - root - Metadata standard: croissant_pangaea_standard (structured output)
2026-07-02 12:09:17,133 - INFO - root - ==========

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:882: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-02 12:11:26,243 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 12:11:26,277 - WARNING - root - Plan validation warning: Step 6 ('generate_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-02 12:11:26,277 - INFO - root - Plan generated successfully!
2026-07-02 12:11:26,278 - INFO - root - Number of steps: 7
2026-07-02 12:11:26,278 - INFO - root -   Step 1: get_columns_with_first_row (player: dataset_schema_preview)
2026-07-02 12:11:26,278 - INFO - root -   Step 2: get_context_overview (player: relationship_analyst)
2026-07-02 12:11:26,279 - INFO - root -   Step 3: detect_temporal_columns (player: spatial_temporal_specialist)
2026-07-02 12:11:26,279 - INFO - root -   Step 4: detect_spatial_columns (player: spatial_temporal_specialist)
2026-07-02 12:11:26,279 - INFO - root -   Step 5: get_spatial_extent_from_tuple_column (player: spatial_temporal_specialist)
2026-07-02 12:11:26,279 -

...

2026-07-02 13:03:07,027 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:07,029 - INFO - root - Player 'relationship_analyst_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-07-02 13:03:07,029 - INFO - root - Player 'relationship_analyst_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-07-02 13:03:07,029 - INFO - root -   Player 'relationship_analyst_1' completed execution
2026-07-02 13:03:07,029 - INFO - root -     Output: I will start by getting the context overview to understand the full scope of the data available in the `pangea_995327` context.  
2026-07-02 13:03:07,030 - INFO - root - Single player, skipping debate
2026-07-02 13:03:07,030 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-07-02 13:03:07,644 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:07,645 - INFO 

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:882: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-02 13:03:29,968 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:29,969 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-02 13:03:31,443 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:31,449 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 1 tool(s): detect_temporal_columns
2026-07-02 13:03:32,339 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:32,348 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_spatial_columns
2026-07-02 13:03:35,087 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:35,089 -

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:882: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-02 13:03:41,434 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:41,436 - INFO - root -   Player 'croissant_pangaea_metadata_generator_1' completed execution
2026-07-02 13:03:41,436 - INFO - root -     Output: {   "name": "Lanaittu Core Sediment Mineral Composition and Heavy Mineral Analysis",   "description": "Dataset containing mineralogical composition data from sediment samples collected at the Lanaittu...
2026-07-02 13:03:41,437 - INFO - root - Single player, skipping debate
2026-07-02 13:03:41,437 - INFO - root - --- STEP 4: SYNTHESIS ---
2026-07-02 13:03:41,438 - INFO - root -   Using structured output with schema: CroissantPangaeaMetadata
2026-07-02 13:03:44,704 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-02 13:03:44,708 - INFO - root -   Structured output validated successfully
2026-07-02 13:03:44,709 - INFO - root -   Synthesis complete